# WP1 — U-Net Baseline (UrbanFormer-Mid)

**Question.** Can a rasterized CNN predict the mid-canopy streamwise velocity field
`U_mid = u(x, y, h_m/2) / u_ref` directly from building geometry, on a fixed
78x78 horizontal plane?

**Headline.** A 3-level U-Net reaches **R2 = 0.7194** (RMSE 0.8217, MAE 0.4821)
over fluid cells on the held-out test split (785 cases). This is the raster
baseline the object-based UrbanFormer models are measured against.

**Per-case contract.**
- Inputs (4 channels): `height_map`, `footprint_mask`, `x_grid`, `y_grid`.
- Target: `U_mid` (already normalized by `u_ref`).
- Loss / eval mask: `fluid_mask_mid` (1 = fluid, 0 = building). Building cells
  never enter the loss or the metrics.

The reusable pieces (dataset, model, loss, metrics) live in the `urbanformer`
package; this notebook loads data, trains, evaluates, and plots.

## 1. Setup

In [ ]:
import copy
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from urbanformer.data import UNetMidDataset
from urbanformer.models.unet import UNetMid
from urbanformer.losses import masked_mse
from urbanformer.metrics import field_metrics, per_case_rmse

# --- paths (repo-relative; WP0 writes these) ---
DATA_ROOT  = Path("data/processed")
SPLITS_DIR = Path("data/splits")
CKPT_PATH  = "wp1_unet_best.pt"

# --- training hyperparameters ---
BATCH_SIZE = 8
NUM_EPOCHS = 50
LR         = 1e-3
PATIENCE   = 10          # early-stop after this many epochs without val improvement

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Data

`UNetMidDataset` returns one case as `(x, y, mask)` with the 4-channel raster
input, the target field, and the fluid mask. Splits are the fixed WP0 layout
split (70 / 15 / 15 by whole urban layout, so no case leaks across splits).

In [ ]:
def load_split(split_file, data_root):
    """Read a split .txt of case names into a list of case directories."""
    with open(split_file) as f:
        names = [line.strip() for line in f if line.strip()]
    return [data_root / name for name in names]

train_dirs = load_split(SPLITS_DIR / "train_cases.txt", DATA_ROOT)
val_dirs   = load_split(SPLITS_DIR / "val_cases.txt",   DATA_ROOT)
test_dirs  = load_split(SPLITS_DIR / "test_cases.txt",  DATA_ROOT)
print(f"train/val/test: {len(train_dirs)} / {len(val_dirs)} / {len(test_dirs)}")

train_ds = UNetMidDataset(train_dirs)
val_ds   = UNetMidDataset(val_dirs)
test_ds  = UNetMidDataset(test_dirs)

pin = (DEVICE.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

xb, yb, mb = next(iter(train_loader))
print("batch:", tuple(xb.shape), tuple(yb.shape), tuple(mb.shape))

## 3. Model

A 3-level encoder/decoder U-Net. The encoder halves the plane three times, giving
the odd pyramid **78 -> 39 -> 19 -> 9**; on the way up each transposed conv is
resampled back onto its skip connection with `F.interpolate`, so the odd sizes
round-trip cleanly instead of drifting by a pixel per level.

In [ ]:
model = UNetMid().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNetMid parameters: {n_params:,}")

# forward-pass shape check on a throwaway batch
with torch.no_grad():
    probe = model(torch.zeros(2, 4, 78, 78, device=DEVICE))
print("forward:", tuple(probe.shape))   # expect (2, 1, 78, 78)

## 4. Training

The loss is `masked_mse`: mean squared error over fluid cells only, so building
cells contribute nothing to the gradient. We keep the best validation checkpoint
and early-stop after `PATIENCE` epochs without improvement.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_loss    = float("inf")
best_weights     = None
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    # --- train ---
    model.train()
    train_loss = 0.0
    for x, y, mask in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        x, y, mask = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x).squeeze(1)
        loss = masked_mse(pred, y, mask)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y, mask in val_loader:
            x, y, mask = x.to(DEVICE), y.to(DEVICE), mask.to(DEVICE)
            val_loss += masked_mse(model(x).squeeze(1), y, mask).item()
    val_loss /= len(val_loader)

    print(f"epoch {epoch + 1}/{NUM_EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f}")

    # --- checkpoint + early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, CKPT_PATH)
        patience_counter = 0
        print(f"  -> new best ({val_loss:.4f}); saved {CKPT_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  -> early stop at epoch {epoch + 1}")
            break

## 5. Evaluation

Results-first: over the held-out test split the U-Net reaches **R2 = 0.7194**
(RMSE 0.8217, MAE 0.4821) on fluid cells. All metrics come from
`urbanformer.metrics`, so WP1 and every later WP are scored by identical code.

In [ ]:
model.load_state_dict(best_weights)
model.eval()

preds, targets, masks = [], [], []
with torch.no_grad():
    for x, y, mask in tqdm(test_loader, desc="evaluating"):
        preds.append(model(x.to(DEVICE)).squeeze(1).cpu())   # back to CPU to keep GPU memory flat
        targets.append(y)
        masks.append(mask)

P = torch.cat(preds,   dim=0)   # (N_test, 78, 78)
T = torch.cat(targets, dim=0)
M = torch.cat(masks,   dim=0)

m = field_metrics(P, T, M)
print(f"Test RMSE {m['RMSE']:.4f} | MAE {m['MAE']:.4f} | R2 {m['R2']:.4f} | rel-L2 {m['relL2']:.4f}")

In [ ]:
# Per-case RMSE (fluid-only) drives the representative-case plots below.
case_rmse = per_case_rmse(P, T, M)
best_idx    = int(np.nanargmin(case_rmse))
worst_idx   = int(np.nanargmax(case_rmse))
outlier_idx = int(np.argsort(case_rmse)[int(0.95 * len(case_rmse))])   # 95th-percentile RMSE
print(f"best    {best_idx:4d}  RMSE={case_rmse[best_idx]:.4f}")
print(f"worst   {worst_idx:4d}  RMSE={case_rmse[worst_idx]:.4f}")
print(f"outlier {outlier_idx:4d}  RMSE={case_rmse[outlier_idx]:.4f}")

## 6. Visualization

Ground truth, prediction, and absolute error for the best / worst / outlier
cases. The colormap pins green to `u = 0`; buildings cut by the mid-plane are
blacked out. These plotting helpers are notebook-only presentation code.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def plot_simulation_u(ax, u_norm, solid_mask, title="", vmin=-1.0, vmax=2.5):
    """Plot the normalized mid-plane streamwise velocity u/u_tau."""
    colors = [(0, 0, 0.5), (0, 0.8, 0.8), (0, 0.25, 0), (0.7, 0.7, 0), (0.4, 0, 0)]
    positions = [0, 0.15, (0 - vmin) / (vmax - vmin), 0.55, 1]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom", list(zip(positions, colors)))

    plot_data = u_norm.copy()
    plot_data[solid_mask] = 0
    cax = ax.imshow(plot_data, cmap=cmap, interpolation="bicubic", vmin=vmin, vmax=vmax, origin="upper")
    ax.imshow(np.where(solid_mask, 0, np.nan), cmap="gray", interpolation="nearest", origin="upper", zorder=2)

    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_aspect("equal"); ax.set_title(title); ax.grid(False)
    cbar = plt.colorbar(cax, ax=ax, orientation="horizontal", fraction=0.046, pad=0.15, extend="both")
    cbar.set_label(r"$\overline{u}/u\tau$", fontsize=14)
    return cax


def plot_case(idx, title):
    pred, target, mask = P[idx].numpy(), T[idx].numpy(), M[idx].numpy()
    solid = mask == 0
    error = np.abs(pred - target) * mask

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{title}  |  RMSE={case_rmse[idx]:.4f}", fontsize=13)
    plot_simulation_u(axes[0], target, solid, title="CFD (ground truth)", vmin=-1.0, vmax=3.0)
    plot_simulation_u(axes[1], pred,   solid, title="U-Net prediction",   vmin=-1.0, vmax=3.0)
    im = axes[2].imshow(error, cmap="hot", origin="upper"); axes[2].set_title("|error|")
    plt.colorbar(im, ax=axes[2], orientation="horizontal", fraction=0.046, pad=0.15)
    plt.tight_layout(); plt.show()

In [ ]:
plot_case(best_idx,    "Best case")
plot_case(worst_idx,   "Worst case")
plot_case(outlier_idx, "Outlier case")